In [14]:
import pandas as pd
from sklearn.metrics import (
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBClassifier
import mlflow

In [6]:
from pathlib import Path

DATA_PATH = Path("./Data")
RAW_DATA_PATH = DATA_PATH / "raw" / "Fraud_Data.csv"
PROCESSED_DATA_PATH = DATA_PATH / "processed"

PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

In [7]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Fraud Detection") 

<Experiment: artifact_location='file:C:/Users/trixr/Desktop/FraudDetection/mlflow-data/artifacts/1', creation_time=1781179797941, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1781179797941, lifecycle_stage='active', name='Fraud Detection', tags={}, trace_location=None, workspace='default'>

In [8]:
import mlflow.sklearn 

mlflow.xgboost.autolog()
mlflow.lightgbm.autolog()
mlflow.sklearn.autolog()

# Load Data

In [10]:
X_train = pd.read_csv(PROCESSED_DATA_PATH / "4.1_X_train.csv").drop(columns=["source", "browser", "sex"])
y_train = pd.read_csv(PROCESSED_DATA_PATH / "4.2_y_train.csv")
X_test = pd.read_csv(PROCESSED_DATA_PATH / "4.3_X_test.csv").drop(columns=["source", "browser", "sex"])
y_test = pd.read_csv(PROCESSED_DATA_PATH / "4.4_y_test.csv")

In [11]:
X_train.head()

,purchase_value,age,time_velocity,ip_user_share_count,device_user_share_count,day,hour,source_Ads,source_Direct,source_SEO,browser_Chrome,browser_FireFox,browser_IE,browser_Opera,browser_Safari,sex_F,sex_M
0,14,38,7212744.0,10,10,25,11,True,False,False,True,False,False,False,False,True,False
1,14,38,1.0,10,10,1,0,True,False,False,True,False,False,False,False,True,False
2,14,38,1.0,10,10,1,0,True,False,False,True,False,False,False,False,True,False
3,14,38,1.0,10,10,1,0,True,False,False,True,False,False,False,False,True,False
4,14,38,1.0,10,10,1,0,True,False,False,True,False,False,False,False,True,False


In [12]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120889 entries, 0 to 120888
Data columns (total 17 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   purchase_value           120889 non-null  int64  
 1   age                      120889 non-null  int64  
 2   time_velocity            120889 non-null  float64
 3   ip_user_share_count      120889 non-null  int64  
 4   device_user_share_count  120889 non-null  int64  
 5   day                      120889 non-null  int64  
 6   hour                     120889 non-null  int64  
 7   source_Ads               120889 non-null  bool   
 8   source_Direct            120889 non-null  bool   
 9   source_SEO               120889 non-null  bool   
 10  browser_Chrome           120889 non-null  bool   
 11  browser_FireFox          120889 non-null  bool   
 12  browser_IE               120889 non-null  bool   
 13  browser_Opera            120889 non-null  bool   
 14  brow

In [17]:
with mlflow.start_run(run_name="eval XGBoost with CV"):
    # --- 1. Initialize Time Series Split ---
    # n_splits=5 means it will create 5 expanding windows of train/validation pairs
    tscv = TimeSeriesSplit(n_splits=5)
    
    # Lists to store metrics for each fold
    fold_roc_aucs = []
    fold_pr_aucs = []
    fold_f1s = []
    
    print("--- Starting Time-Based Cross-Validation ---")
    
    # --- 2. Cross-Validation Loop ---
    # Important: X_train and y_train must remain in their chronological order here
    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train)):
        # Split the data based on chronological indices
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
        # Recalculate scale_pos_weight for the specific training fold to avoid leakage
        scale_pos_weight_fold = y_tr.value_counts()[0] / y_tr.value_counts()[1]
    
        # Initialize model with your specific hyperparameters
        xgb_fold = XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight_fold,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        )
    
        # Fit on fold training data, using the fold validation data for early stopping
        xgb_fold.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
    
        # Predict on validation fold
        y_val_prob = xgb_fold.predict_proba(X_val)[:, 1]
        y_val_pred = xgb_fold.predict(X_val)
        # Calculate metrics
        
        roc_auc = roc_auc_score(y_val, y_val_prob)
        pr_auc = average_precision_score(y_val, y_val_prob)
        f1 = f1_score(y_val, y_val_pred)
        
    
        fold_roc_aucs.append(roc_auc)
        fold_pr_aucs.append(pr_auc)
        fold_f1s.append(f1)
    
        print(f"Fold {fold + 1} -> F1: {f1:.4f} | ROC-AUC: {roc_auc:.4f} | PR-AUC: {pr_auc:.4f}")
    
    print("\n--- CV Results Summary ---")
    print(f"Mean ROC-AUC: {np.mean(fold_roc_aucs):.4f} (+/- {np.std(fold_roc_aucs):.4f})")
    print(f"Mean PR-AUC:  {np.mean(fold_pr_aucs):.4f} (+/- {np.std(fold_pr_aucs):.4f})")
    print(f"Mean F1:  {np.mean(fold_f1s):.4f} (+/- {np.std(fold_f1s):.4f})")
    mlflow.log_metric("Mean ROC-AUC", np.mean(fold_roc_aucs))
    mlflow.log_metric("Mean PR-AUC", np.mean(fold_pr_aucs))
    mlflow.log_metric("Mean F1", np.mean(fold_f1s))
    print("-" * 40)



--- Starting Time-Based Cross-Validation ---


2026/06/12 02:49:38 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\trixr\Desktop\FraudDetection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/06/12 02:49:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/12 02:49:49 WARNING mlfl

Fold 1 -> F1: 0.0613 | ROC-AUC: 0.6592 | PR-AUC: 0.1368


2026/06/12 02:49:55 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during xgboost autologging: The following failures occurred while performing one or more logging operations: [MlflowException('Failed to perform one or more operations on the run with ID fb77bf7e9bb644cb9ce0dac8dd0ac3ed. Failed operations: [RestException("INVALID_PARAMETER_VALUE: Changing param values is not allowed. Params were already logged=\'[{\'key\': \'scale_pos_weight\', \'old_value\': \'1.46320293398533\', \'new_value\': \'3.411758265820013\'}]\' for run ID=\'fb77bf7e9bb644cb9ce0dac8dd0ac3ed\'.")]')]
2026/06/12 02:49:55 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\trixr\Desktop\FraudDetection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as

Fold 2 -> F1: 0.1512 | ROC-AUC: 0.6557 | PR-AUC: 0.1271


...

Fold 3 -> F1: 0.2474 | ROC-AUC: 0.6575 | PR-AUC: 0.1458


2026/06/12 02:50:29 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during xgboost autologging: The following failures occurred while performing one or more logging operations: [MlflowException('Failed to perform one or more operations on the run with ID fb77bf7e9bb644cb9ce0dac8dd0ac3ed. Failed operations: [RestException("INVALID_PARAMETER_VALUE: Changing param values is not allowed. Params were already logged=\'[{\'key\': \'scale_pos_weight\', \'old_value\': \'1.46320293398533\', \'new_value\': \'6.352705045160113\'}]\' for run ID=\'fb77bf7e9bb644cb9ce0dac8dd0ac3ed\'.")]')]
2026/06/12 02:50:29 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\trixr\Desktop\FraudDetection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as

Fold 4 -> F1: 0.2668 | ROC-AUC: 0.6705 | PR-AUC: 0.1483


2026/06/12 02:50:49 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during xgboost autologging: The following failures occurred while performing one or more logging operations: [MlflowException('Failed to perform one or more operations on the run with ID fb77bf7e9bb644cb9ce0dac8dd0ac3ed. Failed operations: [RestException("INVALID_PARAMETER_VALUE: Changing param values is not allowed. Params were already logged=\'[{\'key\': \'scale_pos_weight\', \'old_value\': \'1.46320293398533\', \'new_value\': \'7.50278528021607\'}]\' for run ID=\'fb77bf7e9bb644cb9ce0dac8dd0ac3ed\'.")]')]
2026/06/12 02:50:49 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\trixr\Desktop\FraudDetection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as 

Fold 5 -> F1: 0.2895 | ROC-AUC: 0.6697 | PR-AUC: 0.1580

--- CV Results Summary ---
Mean ROC-AUC: 0.6625 (+/- 0.0063)
Mean PR-AUC:  0.1432 (+/- 0.0105)
Mean F1:  0.2032 (+/- 0.0852)
----------------------------------------
🏃 View run eval XGBoost with CV at: http://localhost:5000/#/experiments/1/runs/fb77bf7e9bb644cb9ce0dac8dd0ac3ed
🧪 View experiment at: http://localhost:5000/#/experiments/1


# Save CSV

In [ ]:
# X_train_resampled.to_csv(PROCESSED_DATA_PATH / "5.1_X_train.csv", index=False)
# y_train_resampled.to_csv(PROCESSED_DATA_PATH / "5.2_y_train.csv", index=False)
